In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".")))

from modules.data_loader import load_config, load_ground_truth, load_experiment_json
from modules.kbrs_metrics import get_official_metrics_df
from modules.visualization import feature

In [2]:
cfg = load_config()

In [3]:
fold1_test_replays = [275, 1725, 3613, 4520, 4664]

In [4]:
gt_df = load_ground_truth(cfg["data"]["label_dst_dir"], fold1_test_replays, "all_correct")

In [5]:
# # IoU 계산 함수 정의 (입력 포맷: [x, y, w, h])
# def calculate_iou(boxA, boxB):
#     xA = max(boxA[0], boxB[0])
#     yA = max(boxA[1], boxB[1])
#     xB = min(boxA[0]+boxA[2], boxB[0]+boxB[2])
#     yB = min(boxA[1]+boxA[3], boxB[1]+boxB[3])
    
#     interArea = max(0, xB - xA) * max(0, yB - yA)
#     boxAArea = boxA[2] * boxA[3]
#     boxBArea = boxB[2] * boxB[3]
#     unionArea = boxAArea + boxBArea - interArea
    
#     return interArea / unionArea if unionArea > 0 else 0

In [6]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm  # Jupyter Notebook용 tqdm

def calculate_temporal_alignment_batch(input_df, window_size=12, majority_threshold=3):
    """
    데이터프레임 전체를 입력받아 시간적 관대함(Temporal Window)이 적용된 
    temporal_alignment_score를 고속으로 계산합니다.
    """
    df = input_df.copy()
    
    # 1. 사전 전처리: 원본 데이터에 annotator_id 나 좌표 컬럼이 없을 경우를 대비해 생성
    if 'annotator_id' not in df.columns:
        df['annotator_id'] = df.groupby(['replay', 'image_id']).cumcount() + 1
        
    if 'x' not in df.columns:
        df[['x', 'y', 'w', 'h']] = pd.DataFrame(df['bbox'].tolist(), index=df.index)
        
    results = []
    
    # 리플레이별로 그룹화
    replays = df.groupby('replay')
    
    for replay_name, replay_df in replays:
        # 프레임(image_id) 순으로 정렬
        replay_df = replay_df.sort_values('image_id')
        
        # 캔버스 크기를 고정하기 위해 해당 리플레이의 최대/최소 좌표 추출 (정수형 강제)
        min_x = int(replay_df['x'].min())
        min_y = int(replay_df['y'].min())
        max_x = int((replay_df['x'] + replay_df['w']).max())
        max_y = int((replay_df['y'] + replay_df['h']).max())
        
        width = max_x - min_x
        height = max_y - min_y
        
        if width <= 0 or height <= 0:
            continue
            
        # 해당 리플레이에 존재하는 모든 유니크한 프레임 번호
        all_frames = sorted(replay_df['image_id'].unique())
        
        # 2. tqdm 적용: 리플레이별로 프레임 탐색 진행률을 표시합니다.
        for target_frame in tqdm(all_frames, desc=f"Replay [{replay_name}] Temporal Window"):
            
            # 윈도우(앞뒤 관대함 구간) 내의 데이터 추출
            window_mask = (replay_df['image_id'] >= target_frame - window_size) & \
                          (replay_df['image_id'] <= target_frame + window_size)
            window_df = replay_df[window_mask]
            
            # 윈도우 내 개인별 누적 시야 마스크 생성
            annotator_masks = {annot_id: np.zeros((height, width), dtype=np.uint8) 
                               for annot_id in window_df['annotator_id'].unique()}
            
            for _, row in window_df.iterrows():
                # 좌표를 안전하게 정수(int)로 변환
                x_start = int(row['x'] - min_x)
                x_end = int(row['x'] + row['w'] - min_x)
                y_start = int(row['y'] - min_y)
                y_end = int(row['y'] + row['h'] - min_y)
                
                # 시야 누적 (1로 덮어쓰기)
                annotator_masks[row['annotator_id']][y_start:y_end, x_start:x_end] = 1
                
            # 시간적 합의(Temporal Consensus) 히트맵 도출
            if not annotator_masks:
                continue
                
            temporal_canvas = sum(annotator_masks.values())
            hotspot_mask = (temporal_canvas >= majority_threshold)
            
            # 현재 프레임(target_frame)의 작업자별 점수 계산
            target_df = replay_df[replay_df['image_id'] == target_frame]
            
            for _, row in target_df.iterrows():
                x_start = int(row['x'] - min_x)
                x_end = int(row['x'] + row['w'] - min_x)
                y_start = int(row['y'] - min_y)
                y_end = int(row['y'] + row['h'] - min_y)
                
                my_view_in_hotspot = hotspot_mask[y_start:y_end, x_start:x_end]
                my_area = row['w'] * row['h']
                
                overlap_area = np.sum(my_view_in_hotspot)
                alignment_score = overlap_area / my_area if my_area > 0 else 0
                
                results.append({
                    'id': row['id'],  # 원본 데이터와의 병합을 위한 고유 ID
                    'replay': replay_name,
                    'image_id': target_frame,
                    'annotator_id': row['annotator_id'],
                    'temporal_alignment_score': round(alignment_score, 4)
                })

    # 전체 결과를 DataFrame으로 반환
    return pd.DataFrame(results)

In [9]:
def analyze_result(results_df):
    desc_df = results_df[["replay", "annotator_id", "temporal_alignment_score"]].groupby(["replay", "annotator_id"]).describe()

    # 다중 인덱스('alignment_score') 제거 및 인덱스 리셋
    desc_df.columns = desc_df.columns.droplevel(0) 
    desc_df = desc_df.reset_index()

    
    # 관전자별로 각 통계치의 평균
    annotator_overall = desc_df.groupby("annotator_id").mean(numeric_only=True)
    
    overall_summary = desc_df.drop(columns=['replay', 'annotator_id']).describe()

    return desc_df, annotator_overall, overall_summary

In [19]:
instant_m3_results_df = calculate_temporal_alignment_batch(gt_df, window_size=0, majority_threshold=3)
instant_m3_desc_df, instant_m3_annotator_overall, instant_m3_overall_summary = analyze_result(instant_m3_results_df)

Replay [1725.rep] Temporal Window:   0%|          | 0/18188 [00:00<?, ?it/s]

Replay [275.rep] Temporal Window:   0%|          | 0/66688 [00:00<?, ?it/s]

Replay [3613.rep] Temporal Window:   0%|          | 0/19046 [00:00<?, ?it/s]

Replay [4520.rep] Temporal Window:   0%|          | 0/24103 [00:00<?, ?it/s]

Replay [4664.rep] Temporal Window:   0%|          | 0/20634 [00:00<?, ?it/s]

In [20]:
print("=== instant, majority_threshold=3 ===")
instant_m3_desc_df

=== instant, majority_threshold=3 ===


,replay,annotator_id,count,mean,std,min,25%,50%,75%,max
0,1725.rep,1,18188.0,0.315698,0.368230,0.0,0.0,0.0000,0.7000,1.0
1,1725.rep,2,18188.0,0.285858,0.365813,0.0,0.0,0.0000,0.6500,1.0
2,1725.rep,3,18188.0,0.328279,0.378491,0.0,0.0,0.0000,0.7208,1.0
3,1725.rep,4,18188.0,0.273796,0.360259,0.0,0.0,0.0000,0.6250,1.0
4,1725.rep,5,18188.0,0.306979,0.362211,0.0,0.0,0.0000,0.6708,1.0
5,275.rep,1,66688.0,0.334709,0.366683,0.0,0.0,0.1333,0.7125,1.0
6,275.rep,2,66688.0,0.271260,0.360528,0.0,0.0,0.0000,0.6417,1.0
7,275.rep,3,66688.0,0.349636,0.373143,0.0,0.0,0.1833,0.7333,1.0
8,275.rep,4,66688.0,0.290824,0.359662,0.0,0.0,0.0000,0.6667,1.0
9,275.rep,5,66688.0,0.333295,0.365935,0.0,0.0,0.1250,0.7083,1.0


In [21]:
print(" - 관전자별")
instant_m3_annotator_overall[['mean', 'std', '75%', 'max']]

 - 관전자별


,mean,std,75%,max
annotator_id,,,,
1,0.298964,0.359876,0.66918,1.0
2,0.268322,0.356874,0.62084,1.0
3,0.311874,0.369582,0.69832,1.0
4,0.271628,0.357163,0.63168,1.0
5,0.290149,0.351801,0.63582,1.0


In [22]:
print(" -  전체 데이터셋")
instant_m3_overall_summary

 -  전체 데이터셋


,count,mean,std,min,25%,50%,75%,max
count,25.00000,25.000000,25.000000,25.0,25.0,25.000000,25.000000,25.0
mean,29731.80000,0.288187,0.359059,0.0,0.0,0.017664,0.651168,1.0
std,18971.79767,0.028029,0.010829,0.0,0.0,0.049662,0.052446,0.0
min,18188.00000,0.238373,0.334932,0.0,0.0,0.000000,0.525000,1.0
25%,19046.00000,0.271260,0.352410,0.0,0.0,0.000000,0.625000,1.0
50%,20634.00000,0.285363,0.359737,0.0,0.0,0.000000,0.650000,1.0
75%,24103.00000,0.306979,0.366683,0.0,0.0,0.000000,0.700000,1.0
max,66688.00000,0.349636,0.378491,0.0,0.0,0.183300,0.733300,1.0


In [11]:
instant_m2_results_df = calculate_temporal_alignment_batch(gt_df, window_size=0, majority_threshold=2)
instant_m2_desc_df, instant_m2_annotator_overall, instant_m2_overall_summary = analyze_result(instant_m2_results_df)

Replay [1725.rep] Temporal Window:   0%|          | 0/18188 [00:00<?, ?it/s]

Replay [275.rep] Temporal Window:   0%|          | 0/66688 [00:00<?, ?it/s]

Replay [3613.rep] Temporal Window:   0%|          | 0/19046 [00:00<?, ?it/s]

Replay [4520.rep] Temporal Window:   0%|          | 0/24103 [00:00<?, ?it/s]

Replay [4664.rep] Temporal Window:   0%|          | 0/20634 [00:00<?, ?it/s]

In [17]:
print("=== instant, majority_threshold=2 ===")
instant_m2_desc_df

=== instant, majority_threshold=2 ===


,replay,annotator_id,count,mean,std,min,25%,50%,75%,max
0,1725.rep,1,18188.0,0.598712,0.397125,0.0,0.0000,0.79170,0.916700,1.0
1,1725.rep,2,18188.0,0.529011,0.409189,0.0,0.0000,0.68750,0.904200,1.0
2,1725.rep,3,18188.0,0.595407,0.405332,0.0,0.0000,0.79170,0.941700,1.0
3,1725.rep,4,18188.0,0.528713,0.406707,0.0,0.0000,0.67500,0.908300,1.0
4,1725.rep,5,18188.0,0.573932,0.397356,0.0,0.0000,0.75000,0.916700,1.0
5,275.rep,1,66688.0,0.574117,0.390610,0.0,0.0333,0.73330,0.916700,1.0
6,275.rep,2,66688.0,0.471700,0.418190,0.0,0.0000,0.54170,0.900000,1.0
7,275.rep,3,66688.0,0.620819,0.382666,0.0,0.2500,0.79170,0.941700,1.0
8,275.rep,4,66688.0,0.499775,0.412102,0.0,0.0000,0.63330,0.900000,1.0
9,275.rep,5,66688.0,0.540069,0.400839,0.0,0.0000,0.67500,0.916700,1.0


In [18]:
print(" - 관전자별")
instant_m2_annotator_overall[['mean', 'std', '75%', 'max']]

 - 관전자별


,mean,std,75%,max
annotator_id,,,,
1,0.562415,0.394887,0.914610,1.0
2,0.503612,0.409880,0.900840,1.0
3,0.588665,0.398050,0.927315,1.0
4,0.512569,0.410614,0.905820,1.0
5,0.540185,0.391156,0.891680,1.0


In [14]:
print(" -  전체 데이터셋")
instant_m2_overall_summary

 -  전체 데이터셋


,count,mean,std,min,25%,50%,75%,max
count,25.00000,25.000000,25.000000,25.0,25.000000,25.00000,25.000000,25.0
mean,29731.80000,0.541489,0.400917,0.0,0.014332,0.68859,0.908053,1.0
std,18971.79767,0.038743,0.011281,0.0,0.050548,0.06522,0.018639,0.0
min,18188.00000,0.471700,0.380153,0.0,0.000000,0.54170,0.854200,1.0
25%,19046.00000,0.513916,0.390610,0.0,0.000000,0.65000,0.900000,1.0
50%,20634.00000,0.530146,0.400839,0.0,0.000000,0.67500,0.912500,1.0
75%,24103.00000,0.573932,0.409514,0.0,0.000000,0.73330,0.916700,1.0
max,66688.00000,0.620819,0.418190,0.0,0.250000,0.79170,0.941700,1.0


In [23]:
# 전후 12프레임
temporal_m3_w12_results_df = calculate_temporal_alignment_batch(gt_df, window_size=12, majority_threshold=3)
temporal_m3_w12_desc_df, temporal_m3_w12_annotator_overall, temporal_m3_w12_overall_summary = analyze_result(temporal_m3_w12_results_df)

Replay [1725.rep] Temporal Window:   0%|          | 0/18188 [00:00<?, ?it/s]

Replay [275.rep] Temporal Window:   0%|          | 0/66688 [00:00<?, ?it/s]

Replay [3613.rep] Temporal Window:   0%|          | 0/19046 [00:00<?, ?it/s]

Replay [4520.rep] Temporal Window:   0%|          | 0/24103 [00:00<?, ?it/s]

Replay [4664.rep] Temporal Window:   0%|          | 0/20634 [00:00<?, ?it/s]

In [24]:
print("=== temporal: window=12, majority_threshold=3 ===")
temporal_m3_w12_desc_df

=== temporal: window=12, majority_threshold=3 ===


,replay,annotator_id,count,mean,std,min,25%,50%,75%,max
0,1725.rep,1,18188.0,0.398878,0.400282,0.0,0.0,0.3542,0.8042,1.0
1,1725.rep,2,18188.0,0.349617,0.395292,0.0,0.0,0.0000,0.7792,1.0
2,1725.rep,3,18188.0,0.407182,0.410442,0.0,0.0,0.3333,0.8333,1.0
3,1725.rep,4,18188.0,0.334800,0.393381,0.0,0.0,0.0000,0.7500,1.0
4,1725.rep,5,18188.0,0.380604,0.391659,0.0,0.0,0.2625,0.7917,1.0
5,275.rep,1,66688.0,0.423638,0.398833,0.0,0.0,0.4500,0.8250,1.0
6,275.rep,2,66688.0,0.338207,0.395207,0.0,0.0,0.0000,0.7708,1.0
7,275.rep,3,66688.0,0.446574,0.402952,0.0,0.0,0.5208,0.8500,1.0
8,275.rep,4,66688.0,0.359043,0.394578,0.0,0.0,0.0750,0.7875,1.0
9,275.rep,5,66688.0,0.403507,0.394421,0.0,0.0,0.3750,0.8125,1.0


In [25]:
print(" - 관전자별")
temporal_m3_w12_annotator_overall[['mean', 'std', '75%', 'max']]

 - 관전자별


,mean,std,75%,max
annotator_id,,,,
1,0.382249,0.391799,0.78584,1.0
2,0.333834,0.388632,0.74834,1.0
3,0.397762,0.402898,0.81500,1.0
4,0.339440,0.390424,0.74666,1.0
5,0.361098,0.381046,0.74834,1.0


In [26]:
print(" -  전체 데이터셋")
temporal_m3_w12_overall_summary

 -  전체 데이터셋


,count,mean,std,min,25%,50%,75%,max
count,25.00000,25.000000,25.000000,25.0,25.0,25.000000,25.000000,25.0
mean,29731.80000,0.362877,0.390960,0.0,0.0,0.154168,0.768836,1.0
std,18971.79767,0.034824,0.010646,0.0,0.0,0.167866,0.045168,0.0
min,18188.00000,0.310426,0.370717,0.0,0.0,0.000000,0.666700,1.0
25%,19046.00000,0.336312,0.385119,0.0,0.0,0.000000,0.733300,1.0
50%,20634.00000,0.357869,0.393381,0.0,0.0,0.100000,0.779200,1.0
75%,24103.00000,0.380604,0.396118,0.0,0.0,0.300000,0.795800,1.0
max,66688.00000,0.446574,0.410442,0.0,0.0,0.520800,0.850000,1.0


In [27]:
# 전후 12프레임
temporal_m2_w12_results_df = calculate_temporal_alignment_batch(gt_df, window_size=12, majority_threshold=2)
temporal_m2_w12_desc_df, temporal_m2_w12_annotator_overall, temporal_m2_w12_overall_summary = analyze_result(temporal_m2_w12_results_df)

Replay [1725.rep] Temporal Window:   0%|          | 0/18188 [00:00<?, ?it/s]

Replay [275.rep] Temporal Window:   0%|          | 0/66688 [00:00<?, ?it/s]

Replay [3613.rep] Temporal Window:   0%|          | 0/19046 [00:00<?, ?it/s]

Replay [4520.rep] Temporal Window:   0%|          | 0/24103 [00:00<?, ?it/s]

Replay [4664.rep] Temporal Window:   0%|          | 0/20634 [00:00<?, ?it/s]

In [28]:
print("=== temporal: window=12, majority_threshold=2 ===")
temporal_m2_w12_desc_df

=== temporal: window=12, majority_threshold=2 ===


,replay,annotator_id,count,mean,std,min,25%,50%,75%,max
0,1725.rep,1,18188.0,0.679217,0.384523,0.0,0.44170,0.8708,0.9875,1.0
1,1725.rep,2,18188.0,0.600376,0.409550,0.0,0.00000,0.7875,0.9500,1.0
2,1725.rep,3,18188.0,0.668859,0.397443,0.0,0.33330,0.8542,0.9958,1.0
3,1725.rep,4,18188.0,0.598031,0.408569,0.0,0.00000,0.7792,0.9750,1.0
4,1725.rep,5,18188.0,0.651345,0.388581,0.0,0.31145,0.8333,0.9625,1.0
5,275.rep,1,66688.0,0.667863,0.381025,0.0,0.41250,0.8500,0.9833,1.0
6,275.rep,2,66688.0,0.556466,0.424664,0.0,0.00000,0.7500,0.9708,1.0
7,275.rep,3,66688.0,0.709423,0.368416,0.0,0.55420,0.9000,0.9917,1.0
8,275.rep,4,66688.0,0.573647,0.417762,0.0,0.00000,0.7500,0.9667,1.0
9,275.rep,5,66688.0,0.613496,0.398933,0.0,0.13750,0.8000,0.9792,1.0


In [29]:
print(" - 관전자별")
temporal_m2_w12_annotator_overall[['mean', 'std', '75%', 'max']]

 - 관전자별


,mean,std,75%,max
annotator_id,,,,
1,0.651741,0.384496,0.97000,1.0
2,0.582907,0.410390,0.95582,1.0
3,0.674928,0.386799,0.99168,1.0
4,0.592339,0.411334,0.96668,1.0
5,0.617737,0.385216,0.94918,1.0


In [30]:
print(" -  전체 데이터셋")
temporal_m2_w12_overall_summary

 -  전체 데이터셋


,count,mean,std,min,25%,50%,75%,max
count,25.00000,25.000000,25.000000,25.0,25.000000,25.000000,25.000000,25.0
mean,29731.80000,0.623931,0.395647,0.0,0.182466,0.804172,0.966672,1.0
std,18971.79767,0.041468,0.016434,0.0,0.199990,0.047058,0.020869,0.0
min,18188.00000,0.556466,0.368416,0.0,0.000000,0.733300,0.916700,1.0
25%,19046.00000,0.595341,0.382906,0.0,0.000000,0.779200,0.950000,1.0
50%,20634.00000,0.613496,0.397443,0.0,0.137500,0.791700,0.970800,1.0
75%,24103.00000,0.658731,0.409125,0.0,0.366700,0.833300,0.983300,1.0
max,66688.00000,0.709423,0.424664,0.0,0.554200,0.900000,0.995800,1.0
